In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [5]:
# load data and prep for training
df = pd.read_csv('all_data.csv')
df = df.dropna(subset=['comment_text', 'toxicity']) # drop blank comments
# convert toxicity types to binary classification
df['toxicity_binary'] = (df['toxicity'] >= 0.5).astype(int)
df['severe_toxicity_binary'] = (df['severe_toxicity'] >= 0.5).astype(int)
df['obscene_binary'] = (df['obscene'] >= 0.5).astype(int) 
df['sexual_explicit_binary'] = (df['sexual_explicit'] >= 0.5).astype(int) 
df['identity_attack_binary'] = (df['identity_attack'] >= 0.5).astype(int) 
df['insult_binary'] = (df['insult'] >= 0.5).astype(int) 
df['threat_binary'] = (df['threat'] >= 0.5).astype(int)

# split data and save
train_df = df[df['split'] == 'train']
test_df = df[df['split'] == 'test']
train_df.to_csv('train.csv', index=False)
test_df.to_csv('test.csv', index=False)

In [6]:
# training using scikit learn
train_df = pd.read_csv('train.csv', usecols=['comment_text', 'toxicity_binary'])
test_df = pd.read_csv('test.csv', usecols=['comment_text', 'toxicity_binary'])

vectorizer = TfidfVectorizer(max_features=10000) # limit to 10k words
X_train = vectorizer.fit_transform(train_df['comment_text'])
X_test = vectorizer.transform(test_df['comment_text'])

# Define the target labels
y_train = train_df['toxicity_binary']
y_test = test_df['toxicity_binary']

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print(classification_report(y_test, y_pred))

Accuracy: 0.9467
              precision    recall  f1-score   support

           0       0.95      0.99      0.97    179193
           1       0.78      0.46      0.58     15448

    accuracy                           0.95    194641
   macro avg       0.87      0.72      0.77    194641
weighted avg       0.94      0.95      0.94    194641



In [11]:
# test some comments
test_comments = [
                "everyone should die", 
                "If you cop-suckers can't see a problem with this, then go suck the barrel of a Glock.",
                "NO !  There are no alternative facts. Go check for yourself. It is people like you, who have no idea what you are talking about that has gotten this State and Country into the mess it is in. People who think the Goverment, be it State or Federal, can spend the peoples money better than they can, is stupid and nonsensical. Politicians use taxes as Personal slush accounts to continue their carrers, buying votes from the lame and the lazy.",
                "yep, this crap sounds like its from a libertarian",
                "That is Child Abuse!!!"
                ]

for comment in test_comments:
    comment_transformed = vectorizer.transform([comment])
    prediction = model.predict(comment_transformed)
    print(f"Comment: {comment}")
    print(f"Toxic: {'yes' if prediction[0] == 1 else 'no'}", end="\n\n")

Comment: everyone should die
Toxic: no

Comment: If you cop-suckers can't see a problem with this, then go suck the barrel of a Glock.
Toxic: yes

Comment: NO !  There are no alternative facts. Go check for yourself. It is people like you, who have no idea what you are talking about that has gotten this State and Country into the mess it is in. People who think the Goverment, be it State or Federal, can spend the peoples money better than they can, is stupid and nonsensical. Politicians use taxes as Personal slush accounts to continue their carrers, buying votes from the lame and the lazy.
Toxic: yes

Comment: yep, this crap sounds like its from a libertarian
Toxic: yes

Comment: That is Child Abuse!!!
Toxic: no

